# European Flight Delay Prediction — Pipeline

This notebook implements and tests an Apache Beam data pipeline for the European Flight Delay Prediction project. It reads Parquet flight data generated by '03_data_splitting.ipynb', validates records separating rejected records.

The pipeline will subsequently be extended to support automated preprocessing and reproducible machine-learning datasets.


REMOVE next code block before downloading

In [ ]:
!rm -rf /content/european-flight-delay-prediction

## Prerequisites

The project dependencies are defined in the repository-level `requirements.txt`
file. When this notebook is opened in Google Colab from GitHub, the repository is cloned into the Colab environment
before installing the required dependencies.

In [2]:
import os

# Clone the project repository into the Colab environment
REPO_URL = "https://github.com/sharonjmaye/european-flight-delay-prediction.git"
REPO_DIR = "european-flight-delay-prediction"

%cd /content/
if os.path.exists(REPO_DIR):
    print(f"Repository already exists: {REPO_DIR} will pull latest version")
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    print("Repository cloned successfully.")
    # Change the current working directory to the project root
    %cd {REPO_DIR}



/content
Repository already exists: european-flight-delay-prediction will pull latest version
/content/european-flight-delay-prediction
Already up to date.


## 1. Setup & Environment Installation

The project dependencies should be listed in `requirements.txt`. Run the following cell before continuing. If packages are newly installed, restart the Jupyter kernel if prompted.

In [3]:
%pip install -r requirements.txt

## 2. Configuration & GCP Authentication

Set the project and bucket that the pipeline will use & authenticate to GCP.

Review the folllowing values before running to ensure they are correct for your run:

*   PROJECT_ID
*   BUCKET_NAME
*   SPLIT_RUN_ID - this should reference existing notebook 03 split run


In [4]:

from google.colab import auth
auth.authenticate_user()

from pathlib import Path
from google.cloud import storage
import pandas as pd
import json

PROJECT_ID = "setu-data-handling"
BUCKET_NAME = "european-flight-delay-prediction"

# Existing Notebook 03 split run
SPLIT_RUN_ID = "20261020T065554Z-cc2d29384141"
SPLIT_PREFIX = f"processed/splits/runs/{SPLIT_RUN_ID}"

TARGET_COLUMN = "delayed_15min"

WORK_DIR = Path("/content/flight_preprocessing")
WORK_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_TRAIN = WORK_DIR / "flights_train.parquet"
LOCAL_SAMPLE = WORK_DIR / "flights_sample_100.parquet"

OUTPUT_DIR = WORK_DIR / "beam_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PREFIX = str(OUTPUT_DIR / "flights_sample")

client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_NAME)

# Confirm the source run exists
provenance_blob = bucket.blob(
    f"{SPLIT_PREFIX}/provenance.json"
)

if not provenance_blob.exists():
    raise FileNotFoundError(
        f"Split run not found: gs://{BUCKET_NAME}/{SPLIT_PREFIX}"
    )

source_provenance = json.loads(
    provenance_blob.download_as_text()
)

assert source_provenance["run_id"] == SPLIT_RUN_ID

print("Connected to Google Cloud Storage.")
print("Source split run:", SPLIT_RUN_ID)
print("Source split sizes:", source_provenance["split_rows"])


Connected to Google Cloud Storage.
Source split run: 20261020T065554Z-cc2d29384141
Source split sizes: {'train': 80000, 'dev': 10000, 'test': 10000}


## 3. Import the Pipeline


In [5]:
import sys
import inspect
import os

# Allow imports within the notebook
sys.path.insert(0, os.path.abspath("pipeline"))

# Allow imports in separate Python processes (pytest)
os.environ["PYTHONPATH"] = os.path.abspath("pipeline")

import flight_pipeline as fp

In [9]:
### REMOVE BEFORE DOWNLOAD
import importlib

importlib.reload(fp)

<module 'flight_pipeline' from '/content/european-flight-delay-prediction/pipeline/flight_pipeline.py'>

## 4. Run Automated Tests - RENAME to run automated test with sample valid data


In [10]:
!python -m pytest pipeline/tests -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/european-flight-delay-prediction
plugins: langsmith-0.14.17, typeguard-4.6.0, anyio-4.16.0, platformdirs-4.14.3
collected 1 item                                                               

pipeline/tests/test_pipeline.py::test_valid_and_invalid_rows PASSED      [100%]

============================== 1 passed in 4.42s ===============================
Really destroying service at localhost:40563 with cmd: ('/root/.apache_beam/cache/prism/bin/apache_beam-v2.77.0-prism-linux-amd64', '--job_port', '40563', '--log_level', 'info', '--log_kind', 'json', '--serve_http=false')


## 5. Run Pipeline on Sample Flight data

### Download a sample of 100 records from training data

In [14]:
# Download the training split created by Notebook 03
SOURCE_OBJECT = f"{SPLIT_PREFIX}/flights_train.parquet"

blob = bucket.blob(SOURCE_OBJECT)

if not blob.exists():
    raise FileNotFoundError(
        f"Training file not found: gs://{BUCKET_NAME}/{SOURCE_OBJECT}"
    )

blob.download_to_filename(str(LOCAL_TRAIN))

# Read the first 100 flights for our initial test
sample_df = pd.read_parquet(LOCAL_TRAIN).head(100)

# Save as a separate Parquet file
sample_df.to_parquet(LOCAL_SAMPLE, index=False)

print("Sample records:", len(sample_df))
print("Sample file:", LOCAL_SAMPLE)

display(sample_df.head())

Sample records: 100
Sample file: /content/flight_preprocessing/flights_sample_100.parquet


,id,icao24,flt_id,dof,adep,ades,adep_p,ades_p,registration,model,...,last_seen,version,unix_time,source_month,source_year,duration_min,expected_duration_min,delay_min,delayed_15min,route
0,3031707050241192044,4078ae,EXS66KE,2023-01-08,EGNT,LEAL,,,G-DRTR,737-86N,...,2023-01-08 12:30:45,v2.0.0,1.673171e+09,202301,2023,159.500000,160.250000,-0.750000,0,EGNT-LEAL
1,11421724588017851943,4aca6d,SAS1220,2023-09-13,EKYT,EKRK,EKYT,"EKRK, EKCH, EKCH",SE-RSM,None,...,2023-09-13 18:59:55,v2.0.0,1.694630e+09,202309,2023,31.416667,31.416667,0.000000,0,EKYT-EKRK
2,13447175366594310079,4d2232,RYR7AA,2025-07-19,LIMC,LICT,,,9H-QBX,None,...,2025-07-19 14:38:55,v2.0.0,1.752931e+09,202507,2025,86.750000,90.666667,-3.916667,0,LIMC-LICT
3,7901718741327968292,407688,EZY34BP,2023-10-15,LEAL,EGGW,,,G-UZHT,A320-251N,...,2023-10-16 01:08:05,v2.0.0,1.697411e+09,202310,2023,123.250000,134.333333,-11.083333,0,LEAL-EGGW
4,468791439404541435,4402a3,AUA209N,2023-09-25,LOWW,EDDF,,,OE-LXE,None,...,2023-09-25 10:49:55,v2.0.0,1.695635e+09,202309,2023,60.666667,65.833333,-5.166667,0,LOWW-EDDF


### Run the Apache Beam pipeline

In [15]:
import apache_beam as beam

with beam.Pipeline(
    runner="DirectRunner"
) as pipeline:

    fp.build(
        pipeline,
        input_pattern=str(LOCAL_SAMPLE),
        output_prefix=OUTPUT_PREFIX
    )

print("Apache Beam pipeline execution completed.")

Apache Beam pipeline execution completed.


## 6. Inspect Accepted and Rejected Records

In [16]:
from pathlib import Path

good_files = sorted(OUTPUT_DIR.glob("flights_sample_good*.jsonl"))
bad_files = sorted(OUTPUT_DIR.glob("flights_sample_bad*.jsonl"))

def read_jsonl_files(files):
    records = []

    for file in files:
        with open(file, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    records.append(json.loads(line))

    return pd.DataFrame(records)

good_df = read_jsonl_files(good_files)
bad_df = read_jsonl_files(bad_files)

print("Accepted records:", len(good_df))
print("Rejected records:", len(bad_df))

display(good_df.head())

Accepted records: 100
Rejected records: 0


,id,icao24,flt_id,dof,adep,ades,adep_p,ades_p,registration,model,...,last_seen,version,unix_time,source_month,source_year,duration_min,expected_duration_min,delay_min,delayed_15min,route
0,3031707050241192044,4078ae,EXS66KE,2023-01-08 00:00:00,EGNT,LEAL,,,G-DRTR,737-86N,...,2023-01-08 12:30:45,v2.0.0,1.673171e+09,202301,2023,159.500000,160.250000,-0.750000,0,EGNT-LEAL
1,11421724588017851943,4aca6d,SAS1220,2023-09-13 00:00:00,EKYT,EKRK,EKYT,"EKRK, EKCH, EKCH",SE-RSM,None,...,2023-09-13 18:59:55,v2.0.0,1.694630e+09,202309,2023,31.416667,31.416667,0.000000,0,EKYT-EKRK
2,13447175366594310079,4d2232,RYR7AA,2025-07-19 00:00:00,LIMC,LICT,,,9H-QBX,None,...,2025-07-19 14:38:55,v2.0.0,1.752931e+09,202507,2025,86.750000,90.666667,-3.916667,0,LIMC-LICT
3,7901718741327968292,407688,EZY34BP,2023-10-15 00:00:00,LEAL,EGGW,,,G-UZHT,A320-251N,...,2023-10-16 01:08:05,v2.0.0,1.697411e+09,202310,2023,123.250000,134.333333,-11.083333,0,LEAL-EGGW
4,468791439404541435,4402a3,AUA209N,2023-09-25 00:00:00,LOWW,EDDF,,,OE-LXE,None,...,2023-09-25 10:49:55,v2.0.0,1.695635e+09,202309,2023,60.666667,65.833333,-5.166667,0,LOWW-EDDF


## 7. Reconcile Input and Output Counts

In [17]:
input_count = len(sample_df)
good_count = len(good_df)
bad_count = len(bad_df)

print("Input records:   ", input_count)
print("Accepted records:", good_count)
print("Rejected records:", bad_count)

assert input_count == good_count + bad_count, (
    "Reconciliation failed: input and output counts differ"
)

print("Reconciliation passed.")

Input records:    100
Accepted records: 100
Rejected records: 0
Reconciliation passed.


## 8. Create new dataset containing Invalid Records

In [48]:
# Start with 19 genuine flight records
test_df = sample_df.head(19).copy()

# Create deliberately invalid records
invalid_rows = []

# Test 1: Missing flight ID
row = sample_df.iloc[0].copy()
row["id"] = None
invalid_rows.append(row)

# Test 2: Missing source year
row = sample_df.iloc[1].copy()
row["source_year"] = None
invalid_rows.append(row)

# Test 3: Invalid source year
row = sample_df.iloc[2].copy()
row["source_year"] = 2020
invalid_rows.append(row)

# Test 4: Invalid source year
row = sample_df.iloc[3].copy()
row["source_year"] = 'test'
invalid_rows.append(row)

# Test 5: Missing source month
row = sample_df.iloc[4].copy()
row["source_month"] = None
invalid_rows.append(row)

# Test 6: Invalid source month
row = sample_df.iloc[5].copy()
row["source_month"] = 'test'
invalid_rows.append(row)

# Test 7: Source month and source year mismatch
row = sample_df.iloc[6].copy()
row["source_month"] = '202301'
row["source_year"] = '2024'
invalid_rows.append(row)

# Test 8: Missing flight date
row = sample_df.iloc[7].copy()
row["dof"] = None
invalid_rows.append(row)

# Test 9: Invalid flight date
row = sample_df.iloc[8].copy()
row["dof"] = 'test'
invalid_rows.append(row)

# Test 10: Flight Date and source year mismatch
row = sample_df.iloc[9].copy()
row["dof"] = '2023-01-01'
row["source_month"] = '202401'
row['source_year'] = 2024
invalid_rows.append(row)

# Test 11: Flight Date and source month mismatch
row = sample_df.iloc[10].copy()
row["source_year"] = 2023
row["source_month"] = "202309"
row["dof"] = "2023-08-20"
invalid_rows.append(row)

# Test 12: Missing expected duration min
row = sample_df.iloc[11].copy()
row["expected_duration_min"] = None
invalid_rows.append(row)

# Test 13: Invalid expected duration min
row = sample_df.iloc[12].copy()
row["expected_duration_min"] = 'test'
invalid_rows.append(row)

# Test 14: Invalid expected duration min
row = sample_df.iloc[13].copy()
row["expected_duration_min"] = -1
invalid_rows.append(row)

# Test 15: Invalid target value
row = sample_df.iloc[14].copy()
row["delayed_15min"] = 5
invalid_rows.append(row)

# Test 16: Target value delay min mismatch
row = sample_df.iloc[15].copy()
row["delayed_15min"] = 1
row["delay_min"] = 15
invalid_rows.append(row)

# Test 17: Target value delay min mismatch
row = sample_df.iloc[16].copy()
row["delayed_15min"] = 1
row["delay_min"] = 10
invalid_rows.append(row)

# Test 18: Target value delay min mismatch
row = sample_df.iloc[17].copy()
row["delayed_15min"] = 0
row["delay_min"] = 16
invalid_rows.append(row)

# Combine valid and invalid records
invalid_df = pd.DataFrame(invalid_rows)

test_df = pd.concat(
    [test_df, invalid_df],
    ignore_index=True
)

# Convert object fields to strings so invalid values can be tested
test_df["dof"] = test_df["dof"].astype("string")
test_df["source_year"] = test_df["source_year"].astype("string")
test_df["expected_duration_min"] = test_df["expected_duration_min"].astype("string")

# Preserve large flight identifiers without losing precision
test_df["id"] = pd.array(test_df["id"], dtype="UInt64")

# Save the validation test dataset
INVALID_SAMPLE = WORK_DIR / "flights_validation_test.parquet"
test_df.to_parquet(INVALID_SAMPLE, index=False)

print("Total test records:", len(test_df))
display(test_df[["id", "source_month", "delayed_15min", "dof"]])

Total test records: 37


,id,source_month,delayed_15min,dof
0,3031707050241191936,202301,0,2023-01-08 00:00:00
1,11421724588017852416,202309,0,2023-09-13 00:00:00
2,13447175366594310144,202507,0,2025-07-19 00:00:00
3,7901718741327968256,202310,0,2023-10-15 00:00:00
4,468791439404541440,202309,0,2023-09-25 00:00:00
5,16833618418071527424,202401,0,2024-01-18 00:00:00
6,16974462428384841728,202407,0,2024-07-31 00:00:00
7,15230789142742016000,202304,0,2023-04-11 00:00:00
8,2894706958599154176,202407,0,2024-07-27 00:00:00
9,14065787801493788672,202308,0,2023-08-03 00:00:00


## 9. Execute pipeline against the new test dataset containing invalid records

In [49]:
# Separate output prefix to preserve previous results
VALIDATION_PREFIX = str(
    OUTPUT_DIR / "flights_validation_test"
)

with beam.Pipeline(runner="DirectRunner") as pipeline:

    fp.build(
        pipeline,
        input_pattern=str(INVALID_SAMPLE),
        output_prefix=VALIDATION_PREFIX
    )

print("Validation test pipeline completed.")

Validation test pipeline completed.


## 10. Inspect the quarantined records

In [50]:
validation_good_files = sorted(
    OUTPUT_DIR.glob("flights_validation_test_good*.jsonl")
)

validation_bad_files = sorted(
    OUTPUT_DIR.glob("flights_validation_test_bad*.jsonl")
)

validation_good = read_jsonl_files(validation_good_files)
validation_bad = read_jsonl_files(validation_bad_files)

print("Input records:", len(test_df))
print("Accepted records:", len(validation_good))
print("Rejected records:", len(validation_bad))

if not validation_bad.empty:
    display(validation_bad[["reason", "id"]])


Input records: 37
Accepted records: 19
Rejected records: 18


,reason,id
0,missing_id,None
1,missing_source_year,11421724588017852416
2,invalid_source_year,13447175366594310144
3,invalid_source_year,7901718741327968256
4,missing_source_month,468791439404541440
5,invalid_source_month,16833618418071527424
6,invalid_source_month,16974462428384841728
7,missing_flight_date,15230789142742016000
8,invalid_flight_date,2894706958599154176
9,flight_year_mismatch,14065787801493788672


## 11. Reconcile and verify

In [53]:
input_count = len(test_df)
good_count = len(validation_good)
bad_count = len(validation_bad)

assert input_count == good_count + bad_count, (
    "Reconciliation failed, total count incorrect"
)

assert 19 == good_count, ("Reconciliation failed, good record count incorrect expected 19 but was " + str(good_count))

assert 18 == bad_count, ("Reconciliation failed, bad record count incorrect expected 18 but was " + str(bad_count))

print("Reconciliation passed.")

expected_reasons = {
    "missing_source_year",
    "missing_id",
    "invalid_expected_duration",
    "missing_flight_date",
    "invalid_flight_date",
    "flight_year_mismatch",
    "flight_month_mismatch",
    "missing_source_month",
    "invalid_source_month",
    "invalid_source_year",
    "missing_expected_duration",
    "target_delay_mismatch",
    "invalid_target",
}

actual_reasons = set(
    validation_bad["reason"].tolist()
) if not validation_bad.empty else set()

print("Expected reasons:", expected_reasons)
print("Actual reasons:", actual_reasons)

assert expected_reasons == actual_reasons, (
    f"Unexpected rejection reasons. "
    f"Missing: {expected_reasons - actual_reasons}. "
    f"Unexpected: {actual_reasons - expected_reasons}"
)

print("All expected rejection reasons verified.")

Reconciliation passed.
Expected reasons: {'missing_source_year', 'missing_id', 'invalid_expected_duration', 'missing_flight_date', 'flight_year_mismatch', 'missing_source_month', 'invalid_flight_date', 'invalid_source_month', 'invalid_source_year', 'flight_month_mismatch', 'missing_expected_duration', 'target_delay_mismatch', 'invalid_target'}
Actual reasons: {'missing_source_year', 'missing_id', 'invalid_expected_duration', 'missing_flight_date', 'missing_source_month', 'flight_year_mismatch', 'invalid_flight_date', 'invalid_source_month', 'invalid_source_year', 'flight_month_mismatch', 'missing_expected_duration', 'target_delay_mismatch', 'invalid_target'}
All expected rejection reasons verified.
